# Анализ данных в Python

*Алла Тамбовцева*

## Практикум 1*. Категориальные данные и дискретизация данных в `pandas`

* Категориальные данные в `pandas`
* Дискретизация данных и функция `cut()`

### Категориальные данные в `pandas`

Если в `pandas` нас не устраивает обычный строковый тип `object` (`str`) или числовые метки типа `int` для хранения данных в категориальной шкале, можно задействовать отдельный тип  – `category`. К этому типу можно привести признаки как в номинальной шкале (неупорядоченные значения), так и в порядковой (упорядоченные значения).

Создадим небольшой датафрейм на основе словаря:

In [ ]:
import pandas as pd

In [ ]:
data = pd.DataFrame({"id" : [1, 2, 3, 4, 5, 6, 7], 
              "пол" : ["жен", "муж", "муж", "жен", "жен", "жен", "муж"],
              "город" : ["Москва", "Пермь", "Пермь", "Ярославль", "Москва", 
                        "Москва", "Галич"],
              "v1" : [1, 0, 0, 1, 0, 1, 0], 
              "v2" : ["абсолютно не согласен (не согласна)", 
                      "абсолютно не согласен (не согласна)", 
                      "согласен (согласна)", 
                      "абсолютно согласен (абсолютно согласна)", 
                      "не согласен (не согласна)", 
                      "абсолютно согласен (абсолютно согласна)",
                      "согласен (согласна)"]})
data

Сейчас типы всех столбцов стандартные:

In [ ]:
print(data.dtypes)

Что стоит иметь в виду – Python умеет сравнивать текстовые строки между собой, упорядочивая их по алфавиту. То есть сейчас, если мы, например, сформулируем условие с оператором `>` для столбца `город`, ошибки не будет:

In [ ]:
# все города, которые по алфавиту ниже слова `Москва`

data[data["город"] > "Москва"]

Приведем все категориальные столбцы (кроме id) к типу `category`. Если нас устраивает сортировка по алфавиту, для приведения типа достаточно задействовать метод `.astype()`. Изменим тип столбца `город`:

In [ ]:
data["город"] = data["город"].astype("category")

Внешне ничего не изменилось:

In [ ]:
data

Однако тип столбца уже другой:

In [ ]:
print(data.dtypes)

In [ ]:
print(data["город"])

В конце выдачи со значениями столбца есть указание на то, что здесь четыре категории, а именно `['Галич', 'Москва', 'Пермь', 'Ярославль']`. Никаких содержательных изменений не произошло, города по-прежнему упорядочены по алфавиту. Однако фильтрация со знаком `>` уже не сработает. По умолчанию, если мы нигде не фиксировали, что значения упорядочены, Python понимает, что значения типа `category` сравнивать друг с другом через `>` и `<` уже нельзя:

In [ ]:
# раскомментируйте строку ниже,
# получите ошибку – Unordered Categoricals

# data[data["город"] > "Москва"]

То же можно проделать и с числовыми метками, если есть необходимость. Например, бинарный столбец `v1` тоже можно сделать категориальным:

In [ ]:
data["v1"] = data["v1"].astype("category")

In [ ]:
data["v1"]

Теперь этот признак стал категориальным, однако сами значения не превратились в текст. В выдаче в конце сказано, что здесь две категории, однако их значения – целочисленные (`int64`). Получается, при выборе строк мы по-прежнему сможем использовать целые числа:

In [ ]:
data[data["v1"] == 0]

In [ ]:
data[data["v1"] != 0]

А вот использовать операторы, кроме `=` и `!=`, уже не сможем:

In [ ]:
# раскомментируйте строку ниже,
# получите ошибку – Unordered Categoricals

# data[data["v1"] > 0]

Если категории нужно упорядочить, и порядок отличается от алфавитного (данные в порядковой шкале или собираемся строить модели с определенной базовой категорией), метода `.astype()` не хватит, понадобится функция `Categorical()`. Внутри этой функции помимо самого столбца с данными можно зафиксировать порядок категорий и «включить» упорядочение, если требуется. 

Проделаем разумное упорядочивание для столбца `v2`:

In [ ]:
data["v2"] = pd.Categorical(data["v2"], 
               categories = ["абсолютно не согласен (не согласна)", 
                            "не согласен (не согласна)",
                            "согласен (согласна)",
                            "абсолютно согласен (абсолютно согласна)"], 
              ordered = True)

Проверим тип:

In [ ]:
print(data["v2"])

В выдаче в перечне с уникальными значениями в конце добавились знаки `<`. А значит, теперь для выбора строк можно использовать операторы `<` и `>` (и нестрогие аналоги). Выберем всех согласных (и абсолютно согласных) с некоторым утверждением:

In [ ]:
# алфавит ни на что не влияет
# выбираем все, что по нашему порядку не меньше

data[data["v2"] >= "согласен (согласна)"]

Два нюанса:

* Если в `Categorical()` не добавлять аргумент `categories`, но оставить `ordered = True`, мы получим порядковый признак, но Python сам упорядочит уникальные значения этого столбца по алфавиту. То есть, мы получим то же, что и через `.astype('category')`, только будет разрешены сравнения с `>`, `>=`, `<`, `<=`.

* Если в `categories` мы потеряем какую-то категорию, соответствующие ей значения в столбце будут заменены на пропуски `NaN`. Теоретически, это можно использовать намеренно – если нас интересуют только определенные ответы, можно перечислить их в `categories`, остальные заменятся на `NaN`, а потом при выкидывании мы их удалим.

В заключение посмотрим, что происходит при описании категориальных столбцов:

In [ ]:
print(data.dtypes)

In [ ]:
print(data["город"].describe())

In [ ]:
print(data["v1"].describe())

In [ ]:
print(data["v2"].describe())

У столбцов, которые изначально были текстовыми, ничего нового нет – для типа `category`, как и для `object` (`str`), выводится число заполненных ячеек `count`, число уникальных значений `unique`, мода `top` и частота, соответствующая моде `freq`. А вот со столбцом `v1` интересно. Сам он имеет тип `category` (см выше в `.dtypes`), но его значения имеют тип `int64`. Поэтому значения в столбце по-прежнему целочисленные, но описывается он не как числовой (минимум, максимум, среднее и других стандартные описательные статистик), а как текстовый.

### Дискретизация данных и функция `cut()`

Загрузим данные из файла к основному практикуму `disney-clean.csv` по ссылке и при загрузке выберем только те столбцы, которые нам будут нужны для работы:

In [ ]:
url = "https://raw.githubusercontent.com/allatambov/PyDat25/refs/heads/main/disney_clean.csv"

# если заранее знаем названия нужных столбцов,
# можем указать их в usecols

disney = pd.read_csv(url, usecols = ["Title", "Release year", "IMDB"])
disney

Что такое дискретизация данных? Разбиение непрерывного признака на интервалы, которые можно закодировать конечным набором значений. Превращение количественного непрерывного признака в количественный дискретный признак по заданному правилу. Однако далее мы будем рассматривать не столько дискретизацию в чистом виде, сколько вообще примеры разбиения числового показателя на категории, которые можно упорядочить.

В датафрейме `disney` формально столбец `IMDB` можно считать непрерывным признаком, поскольку это рейтинг после усреднения, и без округления теоретически он может быть со сколько угодно знаками после точки. Посмотрим на его описательные статистики – пригодятся чуть позже:

In [ ]:
disney["IMDB"].describe()

Если мы готовы поступиться точностью (иногда это тоже требуется), нам нужно будет разбить диапазон всех значений рейтинга на интервалы. Сделать это можно с помощью функции `cut()`.

На вход этой функции нужно передать числовой столбец и правило для разбиения – количество равных по размеру интервалов или сами границы интервалов в явном виде. Начнем с простого: попробуем разбить все значения `IMDB` на 5 групп:

In [ ]:
pd.cut(disney["IMDB"], bins = 5)

Функция сделала следующее:

1. Вычислила минимум (1.5) и максимум (8.7) по столбцу, разбила интервал [1.5, 8.7] на 5 равных отрезков:

        (1.493, 2.94], (2.94, 4.38], (4.38, 5.82], (5.82, 7.26], (7.26, 8.7]

   Длина каждого отрезка 1.44 (минимум на самом деле 1.5, а не 1.493, поэтому все ок). Отрезки по умолчанию открыты слева, то есть левая граница в сам отрезок не входит. Так, значение 4.38 будет принадлежать второму отрезку, а не третьему. Такое правило задания границ отрезков объясняет, почему первое значение в первом отрезке не 1.5 – чтобы не потерять минимум, взяв отрезок `(1.5, 2.94]`, Python немного отступает от минимального значения назад.

2. Вернула новый столбец типа `category` из упорядоченных значений 

        (1.493, 2.94] < (2.94, 4.38] < (4.38, 5.82] < (5.82, 7.26] < (7.26, 8.7].
        
Если мы добавим такой столбец в датафрейм, будет не очень удобно:

In [ ]:
disney["IMDB Groups"] = pd.cut(disney["IMDB"], bins = 5)
disney.head()

Поэтому добавим числовые метки для полученных отрезков (порядковая шкала, раз отрезки идут друг за другом на числовой прямой) внутри `cut()`:

In [ ]:
# так лучше

disney["IMDB Groups"] = pd.cut(disney["IMDB"], bins = 5, 
                              labels = [1, 2, 3, 4, 5])
disney.head()

Посмотрим, сколько значений в каждом интервале (в каждой категории):

In [ ]:
# много значений в категории 4 и 5,
# это высокие значения рейтинга

disney["IMDB Groups"].value_counts()

Если нас не устраивает дробный шаг или интервалы одинаковой длины, мы можем самостоятельно зафиксировать границы отрезков:

In [ ]:
# от 1 до 9 с шагом 2

var01 = pd.cut(disney["IMDB"], bins = [1, 3, 5, 7, 9])
var01.value_counts()

In [ ]:
# отрезки (1, 5], (5, 7] и так далее

var02 = pd.cut(disney["IMDB"], bins = [1, 5, 7, 8, 9])
var02.value_counts()

Аналогичным образом попробуем поработать с годом выхода мультфильма – разобьем годы на десятилетия. Попутно решим любопытную задачу – как автоматически определить, с какого десятилетия начать? Определим год выхода самого раннего мультфильма:

In [ ]:
disney["Release year"].min()

Как отсюда получить десятилетие в виде числа `1930`? Вычтем из года остаток от деления на 10:

In [ ]:
min_ = disney["Release year"].min()
start = min_  - min_ % 10
print(start)

Похожим образом определим, на каком десятилетии нужно остановиться:

In [ ]:
disney["Release year"].max()

In [ ]:
max_ = disney["Release year"].max()

# + вперед уходим на 10 лет с запасом
end = max_  - max_ % 10 + 10
print(end)

Чтобы не перечислять годы вручную, воспользуемся функцией `np.arange()` для создания границ интервалов. Функция `range()` не совсем подойдет – год, конечно, по идее целочисленный, но из-за наличия пропусков тип столбца `float`, поэтому нужна функция, умеющая работать с дробными числами:

In [ ]:
import numpy as np

In [ ]:
pd.cut(disney["Release year"], 
       bins = np.arange(start, end + 10, 10))

Добавим вразумительные метки, они могут быть и текстовыми, упорядочение сохранится:

In [ ]:
labs = [str(i) + "s" for i in range(1930, 2030, 10)]
print(labs)

In [ ]:
disney["Release Decade"] = pd.cut(disney["Release year"], 
       bins = np.arange(start, end + 10, 10),
       labels = labs)

disney

P.S. Можете также ознакомиться с [функцией](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.qcut.html) `qcut()`, которая использует статистический подход для разбиения на группы – по квантилям.